# Assessing the uncertainties in the primary-gamma rays (our signal) -- v3

The MC samples have follower particle tracking enabled, which records gamma-rays emitted during the primary (initial) interaction and any secondaries downstream (from Geant4 neutron/proton-nucleus interactions)

The uncertainties are taken from our Empirical model for the p and s-shell states, and applied to signal and (some) background events. Because no GENIE re-weighting exists for the de-excitation model, this is a little ad-hoc but the best I could do.

Why we must apply it to both signal and background:

You might think to yourself "signal should be the only affected population as the nuclear de-excitations we're tagging are coming from NCQE events" - while this is the case a large % of one background category in our sample (out-of-fiducial-volume / OOFV) events contain NCQE events that were mis-reconstructed as occuring within our FV. We therefore must apply the variation to both signal + OOFV background events.

### v3 (2026-10-15): what changed and why

The v2 propagation (`formula_v2`) treated the selected OOFV $\nu$NCQE as a **background** subtracted from the data,
with the scale factor 1.580 of the off-beam CIT configuration. The paper's extraction (Eq. 26) instead **scales the OOFV
$\nu$NCQE with $f$ together with the signal**, so a change of the primary $\gamma$-ray yield now enters only through the
combined selected $\nu$NCQE yield, $\sigma \propto (N_{obs}-M_{bkg}-D_{CIT}-D_{sky})/(M_{sig}+M^{OOFV}_{\nu NCQE})$. In
the v2 configuration the same variation also moved the subtracted background, which inflated it
(p shell +17.3/-5.0% there, about +10.9/-3.2% here).

| | v2 | v3 |
|---|---|---|
| sample | stvPrep flags of the aggregate `xsec_analyzer` file, all 81 bunches | CV selection of 05 (`lib/selection.py`), first 26 bunches, per-file `MC_GENIE-O16` |
| propagation | `formula_v2`: OOFV $\nu$NCQE as background, $f$ = 1.580 | `lib/extraction.py`, paper Eq. 26 (pre-beam CIT, refitted skyshine) |
| T2K-like "others" | -- | new: "other" states emit like the $1s_{1/2}$ hole (population reweighting) |

Unchanged: the primary/secondary $\gamma$ definitions, the line lists, the cat1-cat4 immunity logic, the $E_\gamma$ bands and
the nuisance parameters (`p`, `s_mid`, `s_hi`, `s_norm`) with their uncertainties. The v2 configuration is still evaluated
as a printed cross-check (it reproduces the old numbers).

**Run order:** `python3 08_systematics.py` first (it caches the selection and provides $N_{obs}$, $D_{CIT}$, $D_{sky}$),
then this notebook, then 07. Output: `results/06_primary_gamma.json`.


In [1]:
from lib.constants import *
from lib.imports import *
import json
from lib import paths, selection as SEL, extraction as X
from lib.followers import read_followers

# v3: the CV selection of 05 (vectorised, lib/selection.py), run on the per-file CV MC; cached in cache/mc_sel_Empirical.npz
mc = SEL.mc_selection('Empirical', paths.MC_EMPIRICAL_DIR)

# inputs of the extraction from 08_systematics.py (run it first)
R08 = json.load(open(paths.RESULTS_DIR + '08_systematics.json'))['inputs']
N_OBS, D_CIT, D_SKY = R08['N_obs'], R08['D_CIT'], R08['D_sky']
F_POT = R08['POT_1e20'] / (MC_POT_Empirical / 1e20)
Y_CV = X.yields(mc, F_POT)                 # POT-scaled CV yields per category (first 26 bunches; External x alpha)
F_CV = X.extract_f(Y_CV, N_OBS, D_CIT, D_SKY)
print(f'CV: N_obs {N_OBS}, D_CIT {D_CIT:.2f}, D_sky {D_SKY:.2f}; M_sig {Y_CV["sig"]:.2f}, M_OOFVnu {Y_CV["OOFV_nu"]:.2f}, '
      f'M_bkg {X.m_bkg(Y_CV):.2f} -> f = {F_CV:.4f}')

# rows = CV-selected clusters inside the +-1 sigma bunch window (all 81 bunches; the 26-bunch cut is applied in CELL 1)
rows = mc['in_win']
MC_EVENT_CATEGORY = mc['cat'][rows]      # 05 categories: 0 nuNCQE (FV), 1 NCother, 2 CC, 3 nubarNCQE, 4 OOFV, 5 External
MC_OOFV_NU = mc['oofv_nu'][rows]         # OOFV events that are nuNCQE on 16O (true vertex in the tank, outside the FV)
MC_IN26 = mc['in26'][rows]
TRUTH_FV_RAW = mc['truth'][1]            # all true FV nuNCQE (the efficiency denominator; unchanged by these variations)
assert np.all(mc['w'][rows][(MC_EVENT_CATEGORY == 0) | MC_OOFV_NU] == 1.0)   # TunedCV = 1 for NC: yields = F_POT x counts

# follower records of the selected nuNCQE (FV + OOFV) and of every true FV nuNCQE (population check)
fidx, ce = mc['fidx'][rows], mc['ce'][rows]
nu_rows = np.where((MC_EVENT_CATEGORY == 0) | MC_OOFV_NU)[0]
FOL = read_followers(paths.MC_EMPIRICAL_DIR, list(mc['files']),
                     {i: ce[nu_rows][fidx[nu_rows] == i] for i in np.unique(fidx[nu_rows])})
key = lambda f, e: np.asarray(f, np.int64) * 10_000_000 + np.asarray(e, np.int64)
pos = dict(zip(key(FOL['fidx'], FOL['entry']).tolist(), range(len(FOL['entry']))))
sel_pos = np.array([pos.get(k, -1) for k in key(fidx, ce).tolist()])
HAS_FOLLOWERS = sel_pos >= 0
assert HAS_FOLLOWERS[nu_rows].all()
ix = np.where(HAS_FOLLOWERS, sel_pos, 0)            # non-nuNCQE rows point at a dummy record and are never used below

# same names as v2 (aligned with the selected rows)
MC_FollowerE         = FOL['E'][ix]
MC_FollowerPDG       = FOL['PDG'][ix]
MC_FollowerParentPDG = FOL['parent'][ix]
MC_FollowerT         = FOL['T'][ix]                 # already relative to the true interaction time
MC_Z_nucleon         = FOL['nucleon'][ix]           # proton = 2212, neutron = 2112

print(f'{rows.sum()} selected clusters in the +-1 sigma window ({MC_IN26.sum()} in the first 26 bunches), '
      f'{len(nu_rows)} of them nuNCQE; {FOL["truth_fv"].sum()} true FV nuNCQE read for the population check')

  MC Empirical: selection from cache (3922 files)
CV: N_obs 772, D_CIT 81.29, D_sky 13.95; M_sig 152.61, M_OOFVnu 140.38, M_bkg 248.91 -> f = 1.4603


4211 selected clusters in the +-1 sigma window (1234 in the first 26 bunches), 1656 of them nuNCQE; 19442 true FV nuNCQE read for the population check


In [2]:
"""
═══════════════════════════════════════════
  Branching-ratio systematic propagation
═══════════════════════════════════════════
Ensure the following are in scope:
    MC_Vtx, MC_Vty, MC_Vtz, MC_Vtt, MC_External, MC_FV, MC_NC, MC_QEL,
    MC_MEC, MC_PDG, MC_Z, MC_Z_nucleon, MC_remnant_nucleus   (numpy)
    MC_FollowerT, MC_FollowerE, MC_FollowerPDG, MC_FollowerParentPDG (awkward)
    MC_SIGNAL, MC_EVENT_CATEGORY, MC_SELECTED                (numpy)
"""

# ═══════════════════════════════════════════════════════════════════════════
#  CELL 1 -- Masks (v3)
# ═══════════════════════════════════════════════════════════════════════════
# v3: rows are the CV-selected clusters (05 cuts) in the +-1 sigma bunch window; the first-26-bunch cut of the CV is
# applied here. The selected OOFV nuNCQE keep their v2 name (N_bkg / "bkg") but are NOT a background in v3: they are
# scaled with f together with the FV signal (paper Eq. 26).

mask_signal_reco = (MC_EVENT_CATEGORY == 0) & MC_IN26          # selected FV nuNCQE signal (N')
mask_bkg_reco    = MC_OOFV_NU & MC_IN26                        # selected OOFV nuNCQE

N_total_truth_FV = int(TRUTH_FV_RAW)
N_sig_TOTAL_reco  = int(np.sum(mask_signal_reco))
N_bkg_TOTAL_reco  = int(np.sum(mask_bkg_reco))

print(f"N_total (truth FV NCQE)              = {N_total_truth_FV}")
print(f"N' (reco-selected FV signal)         = {N_sig_TOTAL_reco}")
print(f"N_bkg (reco-selected OOFV nuNCQE)     = {N_bkg_TOTAL_reco}")

# ═══════════════════════════════════════════════════════════════════════════
#  CELL 2 -- Thresholds and ideal-model line lists
# ═══════════════════════════════════════════════════════════════════════════
# These conditions are meant to select "primary" gamma-rays, and have been
# optimized looking at follower particle distributions

T_PRIMARY_NS = 0.001    # |dt| < 1 ps -> primary de-excitation gamma
E_LOW_MEV    = 3.0
E_MID_MEV    = 6.0
E_TOL_MEV    = 0.0005   # tightened match tolerance; loosen if diagnostic
                        # below shows larger float drift in this ntuple

FOLLOWER_E_ALREADY_MEV = True   # enable if followers are in units of MeV (they should be)

# Extracted from levels O16 de-excitation (see Steven Doran's thesis)
P_SHELL_LINES_MEV = [6.32, 9.93, 3.61, 6.18]
S_SHELL_LINES_MEV = [
    3.09, 3.38, 3.68, 3.85, 3.89, 4.44, 4.92, 5.11, 5.69, 5.83,
    6.09, 6.20, 6.44, 6.45, 6.59, 6.73, 6.90, 7.01, 7.03, 7.34,
]
_overlap = [(p, s) for p in P_SHELL_LINES_MEV for s in S_SHELL_LINES_MEV
            if abs(p - s) < E_TOL_MEV]
assert not _overlap, f"p-shell/s-shell line collision: {_overlap}"


# Break up the gammas into different bands as Kobayashi 2006 applies systematics
# in a 3 > E_gamma > 6 MeV, E_gamma > 6 MeV manner
def classify_primary_gamma(E_MeV):
    for line in P_SHELL_LINES_MEV:
        if abs(E_MeV - line) < E_TOL_MEV:
            return ('p', 'hi' if E_MeV > E_MID_MEV else 'mid')
    for line in S_SHELL_LINES_MEV:
        if abs(E_MeV - line) < E_TOL_MEV:
            return ('s', 'hi' if E_MeV > E_MID_MEV else 'mid')
    return None


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 3 -- Event classification, WITH immune-secondary awareness
# ═══════════════════════════════════════════════════════════════════════════
# See Steven Doran's thesis for event categorization following an NCQE interaction
#
# Prompt NCQE topologies can be split into 4 categories:
# 1. Primary gamma only
# 2. Primary and secondary gamma rays
# 3. Secondary gamma ray only
# 4. Neither
#
# Use cat1/cat2/cat3/cat4 distinction from the original R_eff as outlined in Steven's
# thesis analysis. A primary-gamma match alone is NOT sufficient to call an
# event "at risk" under the branching-ratio nuisance parameters -- an event
# that also carries an immune secondary gamma (from FSI of the knocked-out
# nucleon on a neighboring oxygen nucleus) would stay selected even if the
# primary gamma had not fired. Only cat1 events (primary, no immune
# secondary) are genuinely sensitive to this systematic.
#
#   cat1 = has_primary & ~is_immune   -> AT RISK (branching-ratio shiftable)
#   cat2 = has_primary &  is_immune   -> immune (primary + secondary)
#   cat3 = ~has_primary & is_immune   -> immune (secondary only)
#   cat4 = ~has_primary & ~is_immune  -> no gamma signal of any kind
#                                        (We see a lot of these, which I think is due to some tracking issue with the particles)
#                                        (Since we don't have access to those events we ignore them --> not the best but its what I can do with the files I have available)

T_NCAP_NS    = 100.0   # secondary-gamma upper timing bound (excludes
                        # neutron capture)
E_IMMUNE_MEV = 3.0     # Option A: summed secondary energy > this MeV makes
                        # an event immune. 3 MeV is a reasonable detection floor
                         # any secondary gamma-rays below this likely would not be immune,
                          # and rely exclusively on the primary gamma ray to tag


def is_immune_event(E_MeV_all, par, pdg, t, T_prim=T_PRIMARY_NS,
                     T_ncap=T_NCAP_NS, E_immune=E_IMMUNE_MEV):
    sec = (((par == 2112) | (par == 2212)) & (pdg == 22)
           & (t > T_prim) & (t < T_ncap))
    E_sec = E_MeV_all[sec]
    if E_immune == 0.0:
        return bool(np.any(sec))
    return bool(np.sum(E_sec) > E_immune)


def classify_events_with_immunity(f_energy, f_parent_pdg, f_pdg, f_time,
                                    T_prim=T_PRIMARY_NS):
    N = len(f_energy)
    has_primary   = np.zeros(N, bool)
    is_immune     = np.zeros(N, bool)
    event_classes = [[] for _ in range(N)]
    n_unmatched   = 0

    for i in range(N):
        E   = np.asarray(f_energy[i],     dtype=float)
        par = np.asarray(f_parent_pdg[i], dtype=int)
        pdg = np.asarray(f_pdg[i],        dtype=int)
        t   = np.asarray(f_time[i],       dtype=float)

        E_MeV_all = E if FOLLOWER_E_ALREADY_MEV else E * 1000.0

        prim = (par == 0) & (pdg == 22) & (np.abs(t) < T_prim)
        E_prim_MeV = E_MeV_all[prim]

        if E_prim_MeV.size > 0:
            has_primary[i] = True
            for E_g in E_prim_MeV:
                cls = classify_primary_gamma(E_g)
                if cls is None:
                    n_unmatched += 1
                    continue
                event_classes[i].append(cls)

        is_immune[i] = is_immune_event(E_MeV_all, par, pdg, t, T_prim)

    if n_unmatched:
        print(f"  WARNING: {n_unmatched} primary photon(s) did not match "
              f"any known ideal-model line (energy-matching issue, "
              f"distinct from immunity).")

    return has_primary, is_immune, event_classes


def category_counts(has_primary, is_immune):
    return dict(
        cat1=int(np.sum(has_primary & ~is_immune)),
        cat2=int(np.sum(has_primary &  is_immune)),
        cat3=int(np.sum(~has_primary & is_immune)),
        cat4=int(np.sum(~has_primary & ~is_immune)),
    )


def dominant_class_cat1_only(has_primary, is_immune, event_classes):
    """Dominant (shell, band) class -- ONLY for cat1 (at-risk) events.
    Everything else (cat2, cat3, cat4) gets None, i.e. held fixed under any
    branching-ratio nuisance shift."""
    N = len(has_primary)
    out = []
    for i in range(N):
        at_risk = has_primary[i] and not is_immune[i]
        if not at_risk or not event_classes[i]:
            out.append(None)
            continue
        bands = event_classes[i]
        if any(b == 'hi' for _, b in bands):
            shell = next(s for s, b in bands if b == 'hi')
            out.append((shell, 'hi'))
        else:
            out.append(bands[0])
    return np.array(out, dtype=object)


CLASS_LABELS = [('p', 'hi'), ('p', 'mid'), ('s', 'hi'), ('s', 'mid')]

def class_counts(dominant_class):
    return {
        cls: int(np.sum([d == cls for d in dominant_class if d is not None]))
        for cls in CLASS_LABELS
    }


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 4 -- Classify N' (reconstructed signal) and N_bkg (reco background),
#            splitting each into cat1 (at-risk) vs cat2+cat3+cat4 (immune)
# ═══════════════════════════════════════════════════════════════════════════
def build_follower_slices(mask):
    return (
        MC_FollowerE[mask],
        MC_FollowerParentPDG[mask],
        MC_FollowerPDG[mask],
        MC_FollowerT[mask],          # v3: already relative to the interaction time (lib/followers.py)
    )

# --- signal (N') -------------------------------------------------------------
e_s, pp_s, p_s, t_s = build_follower_slices(mask_signal_reco)
has_primary_sig, is_immune_sig, event_classes_sig = classify_events_with_immunity(
    e_s, pp_s, p_s, t_s
)
cats_sig = category_counts(has_primary_sig, is_immune_sig)
dominant_sig = dominant_class_cat1_only(has_primary_sig, is_immune_sig, event_classes_sig)
N_class_signal = class_counts(dominant_sig)          # cat1 only, by class
N_sig_at_risk   = sum(N_class_signal.values())        # = cat1 total
N_sig_immune    = N_sig_TOTAL_reco - N_sig_at_risk    # cat2 + cat3 + cat4,
                                                        # held fixed

# --- background (N_bkg) -------------------------------------------------------
e_b, pp_b, p_b, t_b = build_follower_slices(mask_bkg_reco)
has_primary_bkg, is_immune_bkg, event_classes_bkg = classify_events_with_immunity(
    e_b, pp_b, p_b, t_b
)
cats_bkg = category_counts(has_primary_bkg, is_immune_bkg)
dominant_bkg = dominant_class_cat1_only(has_primary_bkg, is_immune_bkg, event_classes_bkg)
N_class_bkg = class_counts(dominant_bkg)
N_bkg_at_risk = sum(N_class_bkg.values())
N_bkg_immune  = N_bkg_TOTAL_reco - N_bkg_at_risk

N_sig_unmatched = N_sig_immune
N_bkg_unmatched = N_bkg_immune

print("\n" + "=" * 70)
print("  CATEGORY BREAKDOWN (cat1 = at risk; cat2+cat3+cat4 = immune)")
print("=" * 70)
print(f"  {'':<8}{'cat1':>8}{'cat2':>8}{'cat3':>8}{'cat4':>8}{'dilution':>12}")
for label, cats, n_tot in [('signal', cats_sig, N_sig_TOTAL_reco),
                            ('bkg',    cats_bkg, N_bkg_TOTAL_reco)]:
    dilution = 100 * (cats['cat2'] + cats['cat3']) / n_tot if n_tot else 0.0
    print(f"  {label:<8}{cats['cat1']:>8}{cats['cat2']:>8}"
          f"{cats['cat3']:>8}{cats['cat4']:>8}{dilution:>11.1f}%")

print("\n" + "=" * 70)
print("  CLASSIFIED COUNTS (cat1 / at-risk population only)")
print("=" * 70)
print(f"  {'Class':<12}{'N (signal)':>12}{'N (bkg)':>12}")
for cls in CLASS_LABELS:
    print(f"  {cls[0]}-shell {cls[1]:<4}"
          f"{N_class_signal[cls]:>12d}{N_class_bkg[cls]:>12d}")
print(f"  {'immune':<12}{N_sig_immune:>12d}{N_bkg_immune:>12d}")
print(f"\n  NOTE: immune events (cat2+cat3+cat4) are held FIXED -- no "
      f"branching-ratio sigma applied -- since their selection does not "
      f"depend on the primary de-excitation gamma. Only cat1 counts above "
      f"move under the nuisance shifts below.")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 5 -- Relative uncertainties per class (Leuschner 1994 CCIA / p-shell,
#            Kobayashi 2006 / s-shell), stat+syst in quadrature
# ═══════════════════════════════════════════════════════════════════════════
sigma_p_up   = np.sqrt(0.8**2 + 5.4**2)    / 100.0
sigma_p_down = np.sqrt(0.8**2 + 16.4**2)   / 100.0

S_MID_CENTRAL = 27.9
sigma_s_mid_up   = np.sqrt(1.5**2 + 3.4**2) / S_MID_CENTRAL
sigma_s_mid_down = np.sqrt(1.5**2 + 2.6**2) / S_MID_CENTRAL

S_HI_CENTRAL = 15.6
sigma_s_hi_up   = np.sqrt(1.3**2 + 0.6**2) / S_HI_CENTRAL
sigma_s_hi_down = np.sqrt(1.3**2 + 1.0**2) / S_HI_CENTRAL

SIGMA_REL_UP = {
    ('p', 'hi'):  sigma_p_up,  ('p', 'mid'): sigma_p_up,
    ('s', 'hi'):  sigma_s_hi_up, ('s', 'mid'): sigma_s_mid_up,
}
SIGMA_REL_DOWN = {
    ('p', 'hi'):  sigma_p_down,  ('p', 'mid'): sigma_p_down,
    ('s', 'hi'):  sigma_s_hi_down, ('s', 'mid'): sigma_s_mid_down,
}

# Kobayashi 2006: ~10% on S(s1/2) = 1.6 from DWIA with different optical potentials;
# a normalization common to both E_gamma bands
SIGMA_S_NORM = 0.10



# ═══════════════════════════════════════════════════════════════════════════
#  CELL 6 -- Correlated propagation onto the extracted cross section (v3)
# ═══════════════════════════════════════════════════════════════════════════
NUISANCE_PARAMS = ['p', 's_mid', 's_hi', 's_norm']

def class_group(nuisance):
    if nuisance == 'p':      return [('p', 'hi'), ('p', 'mid')]
    if nuisance == 's_mid':  return [('s', 'mid')]
    if nuisance == 's_hi':   return [('s', 'hi')]
    if nuisance == 's_norm': return [('s', 'hi'), ('s', 'mid')]
    raise ValueError(nuisance)

def shift_class_dict(N_class_dict, nuisance, direction):
    # 'up'/'down' = primary gamma YIELD direction (more/fewer gammas), not XS direction
    sign = +1 if direction == 'up' else -1
    out = {}
    for cls, n in N_class_dict.items():
        if cls not in class_group(nuisance):
            out[cls] = float(n)
            continue
        if nuisance == 's_norm':
            sigma = SIGMA_S_NORM
        else:
            sigma = (SIGMA_REL_UP if direction == 'up' else SIGMA_REL_DOWN)[cls]
        out[cls] = n * (1.0 + sign * sigma)
    return out


def shifted_yields(sig, bkg):
    """CV yields with the at-risk (cat1) FV and OOFV nuNCQE counts replaced by the shifted ones; immune events fixed."""
    Y = dict(Y_CV)
    Y['sig']     = Y_CV['sig']     * (sum(sig.values()) + N_sig_immune) / N_sig_TOTAL_reco
    Y['OOFV_nu'] = Y_CV['OOFV_nu'] * (sum(bkg.values()) + N_bkg_immune) / N_bkg_TOTAL_reco
    return Y

# v3 (adopted, paper Eq. 26): sigma = f <sigma_GENIE>, f = (N_obs - M_bkg - D_CIT - D_sky) / (M_sig + M_OOFVnu).
# The true FV nuNCQE count, and hence <sigma_GENIE>, does not change with the branching ratios, so sigma'/sigma = f'/f.
def formula_adopted(sig, bkg):
    return X.extract_f(shifted_yields(sig, bkg), N_OBS, D_CIT, D_SKY)

# v2 configuration, cross-check only: OOFV nuNCQE subtracted as background, off-beam CIT 127.44, skyshine 14.1
# (equivalent to v2's formula_v2 with SCALE_FACTOR = 1.580)
def formula_earlier(sig, bkg):
    return X.extract_f_earlier(shifted_yields(sig, bkg), N_OBS)


def compute_xs_correlated_systematic(N_class_signal, N_class_bkg, formula_fn, verbose=True):
    central = formula_fn(N_class_signal, N_class_bkg)
    frac_up, frac_down = {}, {}
    for nuisance in NUISANCE_PARAMS:
        sig_up   = shift_class_dict(N_class_signal, nuisance, 'up')
        bkg_up   = shift_class_dict(N_class_bkg,    nuisance, 'up')
        sig_down = shift_class_dict(N_class_signal, nuisance, 'down')
        bkg_down = shift_class_dict(N_class_bkg,    nuisance, 'down')
        frac_up[nuisance]   = (formula_fn(sig_up, bkg_up)     - central) / central
        frac_down[nuisance] = (formula_fn(sig_down, bkg_down) - central) / central
        if verbose:
            print(f"  {nuisance:<6s}: {100*frac_up[nuisance]:+.2f}% / {100*frac_down[nuisance]:+.2f}%   (more / fewer gammas)")
    # More gammas -> larger predicted nuNCQE yield -> LOWER extracted XS, so sort each nuisance's shifts by the sign of
    # the XS change rather than by the yield direction
    xs_pos = {n: max(frac_up[n], frac_down[n], 0.0) for n in NUISANCE_PARAMS}
    xs_neg = {n: min(frac_up[n], frac_down[n], 0.0) for n in NUISANCE_PARAMS}
    total_up   = np.sqrt(sum(v**2 for v in xs_pos.values()))
    total_down = np.sqrt(sum(v**2 for v in xs_neg.values()))
    return central, total_up, total_down, {'up': frac_up, 'down': frac_down, 'xs_pos': xs_pos, 'xs_neg': xs_neg}


def shell_totals(per):
    """(up, down) in % for the p shell and for the s shell (s_mid, s_hi, s_norm in quadrature)."""
    q = lambda names, key: 100 * np.sqrt(sum(per[key][n]**2 for n in names))
    return dict(p=(q(['p'], 'xs_pos'), q(['p'], 'xs_neg')),
                s=(q(['s_mid', 's_hi', 's_norm'], 'xs_pos'), q(['s_mid', 's_hi', 's_norm'], 'xs_neg')))


print("\n" + "=" * 70)
print("  CORRELATED FV + OOFV nuNCQE SYSTEMATIC, ADOPTED EXTRACTION (paper Eq. 26)")
print("=" * 70)
central, total_up, total_down, per_nuisance = compute_xs_correlated_systematic(N_class_signal, N_class_bkg, formula_adopted)
SHELL = shell_totals(per_nuisance)
print(f"\n  Central value f = {central:.4f}  (sanity check: should equal the CV f = {F_CV:.4f})")
print(f"  p shell: +{SHELL['p'][0]:.2f}% / -{SHELL['p'][1]:.2f}%    s shell: +{SHELL['s'][0]:.2f}% / -{SHELL['s'][1]:.2f}%")
print(f"  Total (p + s): +{100*total_up:.2f}% / -{100*total_down:.2f}%")

print("\n  cross-check, v2 configuration (OOFV nuNCQE as background, off-beam CIT):")
_, up_old, dn_old, per_old = compute_xs_correlated_systematic(N_class_signal, N_class_bkg, formula_earlier, verbose=False)
SHELL_OLD = shell_totals(per_old)
print(f"  p shell: +{SHELL_OLD['p'][0]:.2f}% / -{SHELL_OLD['p'][1]:.2f}%    s shell: +{SHELL_OLD['s'][0]:.2f}% / "
      f"-{SHELL_OLD['s'][1]:.2f}%   (paper draft 1: p +17.3/-5.0, s +2.5/-2.6)")

# ═══════════════════════════════════════════════════════════════════════════
#  CELL 7 -- Signal-only alternative (no background term), for comparison /
#            as the fallback result while the background piece above is
#            still being validated
# ═══════════════════════════════════════════════════════════════════════════
delta_up_sq   = sum((N_class_signal[c] * SIGMA_REL_UP[c])**2   for c in CLASS_LABELS)
delta_down_sq = sum((N_class_signal[c] * SIGMA_REL_DOWN[c])**2 for c in CLASS_LABELS)
frac_up_sigonly   = np.sqrt(delta_up_sq)   / N_sig_TOTAL_reco
frac_down_sigonly = np.sqrt(delta_down_sq) / N_sig_TOTAL_reco

print("\n" + "=" * 70)
print("  SIGNAL-ONLY SYSTEMATIC (reco-level N', no background term)")
print("=" * 70)
print(f"  N' = {N_sig_TOTAL_reco}")
print(f"  Signal-only systematic: +{100*frac_up_sigonly:.2f}% / "
      f"-{100*frac_down_sigonly:.2f}%")

N_total (truth FV NCQE)              = 19442
N' (reco-selected FV signal)         = 262
N_bkg (reco-selected OOFV nuNCQE)     = 241

  CATEGORY BREAKDOWN (cat1 = at risk; cat2+cat3+cat4 = immune)
              cat1    cat2    cat3    cat4    dilution
  signal       197       6       3      56        3.4%
  bkg          168      14      11      48       10.4%

  CLASSIFIED COUNTS (cat1 / at-risk population only)
  Class         N (signal)     N (bkg)
  p-shell hi           157         144
  p-shell mid            0           0
  s-shell hi            24           7
  s-shell mid           16          17
  immune                65          73

  NOTE: immune events (cat2+cat3+cat4) are held FIXED -- no branching-ratio sigma applied -- since their selection does not depend on the primary de-excitation gamma. Only cat1 counts above move under the nuisance shifts below.

  CORRELATED FV + OOFV nuNCQE SYSTEMATIC, ADOPTED EXTRACTION (paper Eq. 26)
  p     : -3.16% / +10.90%   (more / fewer ga

## T2K-like "other" states and cross-checks (v3)

In [3]:
# ═══════════════════════════════════════════════════════════════════════════
#  T2K-like treatment of the "other" states (v3, new)
# ═══════════════════════════════════════════════════════════════════════════
# The empirical model gives the strength beyond the single-hole states ("other": 33.0% of proton and 35.6% of neutron
# knockouts, paper Table II) no primary gamma ray. T2K's nominal simulation instead integrates these states into the
# 1s1/2 hole (T2K, PRD 100, 112009, Sec. II); no primary gamma is T2K's extreme case (its Sec. V). This cell evaluates
# the T2K nominal assignment as a variation of the CV.
#
# NucDeExcitationSim samples the hole state independently of the interaction kinematics, so the change of the hole
# populations is an EXACT reweighting of the selected events by their primary-gamma class:
#   events with an s-shell primary gamma : (P_s + P_other) / P_s
#   events without a primary gamma      : (P_none - x_s P_other) / P_none
#   events with a p-shell primary gamma : 1
# with x_s the fraction of 1s1/2-hole decays that emit a primary gamma. Unlike CELL 6, ALL selected nuNCQE events are
# reweighted (immune ones included), because the variation changes populations, not the yield of a given state.

TABLE_II = {   # per knockout, % (empirical model, paper Table II)
    2212: dict(p12=14.6, p_gamma=28.0 + 1.7, p_nogamma=2.8, s=20.0, other=33.0),   # proton hole: 6.32, 9.93 / 10.70 MeV
    2112: dict(p12=14.6, p_gamma=29.8,       p_nogamma=0.0, s=20.0, other=35.6),   # neutron hole (isospin symmetry)
}
X_S = 0.435    # fraction of 1s1/2-hole decays with a primary gamma (resolved levels of Kobayashi 2006)
for t in TABLE_II.values():
    t['none'] = t['p12'] + t['p_nogamma'] + t['s'] * (1 - X_S) + t['other']
    t['w_s'] = (t['s'] + t['other']) / t['s']
    t['w_none'] = (t['none'] - X_S * t['other']) / t['none']
    t['s_gamma'] = t['s'] * X_S

def primary_class(has_primary, event_classes):
    """0 = no primary gamma, 1 = p-shell line, 2 = s-shell line only, 3 = primary gamma matching no line (weight 1)."""
    return np.array([0 if not h else 3 if not c else (1 if any(s == 'p' for s, _ in c) else 2)
                     for h, c in zip(has_primary, event_classes)])

cls_sig = primary_class(has_primary_sig, event_classes_sig)
cls_bkg = primary_class(has_primary_bkg, event_classes_bkg)
nuc_sig = np.asarray(MC_Z_nucleon[mask_signal_reco])
nuc_bkg = np.asarray(MC_Z_nucleon[mask_bkg_reco])

def others_weights(cls, nuc):
    w = np.ones(len(cls))
    for n, t in TABLE_II.items():
        w[(nuc == n) & (cls == 2)] = t['w_s']
        w[(nuc == n) & (cls == 0)] = t['w_none']
    return w

w_sig, w_bkg = others_weights(cls_sig, nuc_sig), others_weights(cls_bkg, nuc_bkg)

def f_others(ws, wb, cfg='adopted'):
    Y = dict(Y_CV)
    Y['sig'] = Y_CV['sig'] * ws.sum() / N_sig_TOTAL_reco
    Y['OOFV_nu'] = Y_CV['OOFV_nu'] * wb.sum() / N_bkg_TOTAL_reco
    return X.extract_f(Y, N_OBS, D_CIT, D_SKY) if cfg == 'adopted' else X.extract_f_earlier(Y, N_OBS)

OTHERS = f_others(w_sig, w_bkg) / F_CV - 1
OTHERS_OLD = f_others(w_sig, w_bkg, 'earlier') / X.extract_f_earlier(Y_CV, N_OBS) - 1
rng = np.random.default_rng(20261015)
boot = []
for _ in range(4000):
    ps, pb = rng.poisson(1.0, len(w_sig)), rng.poisson(1.0, len(w_bkg))
    nominal = dict(Y_CV, sig=Y_CV['sig'] * ps.sum() / N_sig_TOTAL_reco, OOFV_nu=Y_CV['OOFV_nu'] * pb.sum() / N_bkg_TOTAL_reco)
    varied = dict(Y_CV, sig=Y_CV['sig'] * (ps * w_sig).sum() / N_sig_TOTAL_reco,
                  OOFV_nu=Y_CV['OOFV_nu'] * (pb * w_bkg).sum() / N_bkg_TOTAL_reco)
    boot.append(X.extract_f(varied, N_OBS, D_CIT, D_SKY) / X.extract_f(nominal, N_OBS, D_CIT, D_SKY) - 1)
OTHERS_MCSTAT = float(np.std(boot))

print('weights: ' + ', '.join(f"{'p' if n == 2212 else 'n'}: s-shell {t['w_s']:.3f}, none {t['w_none']:.3f}" for n, t in TABLE_II.items()))
print(f"selected classes (none / p / s / unmatched): FV {np.bincount(cls_sig, minlength=4)}, OOFV {np.bincount(cls_bkg, minlength=4)}")
print(f"M_sig {Y_CV['sig']:.1f} -> {Y_CV['sig'] * w_sig.sum() / N_sig_TOTAL_reco:.1f}, "
      f"M_OOFVnu {Y_CV['OOFV_nu']:.1f} -> {Y_CV['OOFV_nu'] * w_bkg.sum() / N_bkg_TOTAL_reco:.1f}")
print(f"'others' -> 1s1/2 emission: sigma {100*OTHERS:+.2f}% (+-{100*OTHERS_MCSTAT:.2f}% MC stat.); "
      f"v2 configuration {100*OTHERS_OLD:+.2f}%")

# population check against Table II: true FV nuNCQE, all of them
tf = FOL['truth_fv']
hp_t, imm_t, ec_t = classify_events_with_immunity(FOL['E'][tf], FOL['parent'][tf], FOL['PDG'][tf], FOL['T'][tf])
cls_t, nuc_t = primary_class(hp_t, ec_t), np.asarray(FOL['nucleon'][tf])
POPULATIONS = {}
for n, t in TABLE_II.items():
    m = nuc_t == n
    sim = 100 * np.bincount(cls_t[m], minlength=4)[:3] / m.sum()
    ref = (t['none'], t['p_gamma'], t['s_gamma'])
    POPULATIONS['proton' if n == 2212 else 'neutron'] = dict(simulated=sim.tolist(), table_II=list(ref))
    print(f"{'proton' if n == 2212 else 'neutron'} hole, true FV nuNCQE (N = {m.sum()}): none / p / s = "
          f"{sim[0]:.2f} / {sim[1]:.2f} / {sim[2]:.2f} %  (Table II: {ref[0]:.1f} / {ref[1]:.1f} / {ref[2]:.1f})")

weights: p: s-shell 2.650, none 0.767, n: s-shell 2.780, none 0.748
selected classes (none / p / s / unmatched): FV [ 59 161  42   0], OOFV [ 59 155  27   0]
M_sig 152.6 -> 186.3, M_OOFVnu 140.4 -> 159.3
'others' -> 1s1/2 emission: sigma -15.23% (+-1.97% MC stat.); v2 configuration -24.52%


proton hole, true FV nuNCQE (N = 9626): none / p / s = 61.45 / 29.77 / 8.78 %  (Table II: 61.7 / 29.7 / 8.7)
neutron hole, true FV nuNCQE (N = 9816): none / p / s = 61.83 / 29.65 / 8.53 %  (Table II: 61.5 / 29.8 / 8.7)


In [4]:
# Cross-check (not used in the budget): exact population reweighting of the p-shell strength variation.
# CELL 6 moves only at-risk (cat1) p-shell events and holds events without a primary gamma fixed. Since the strength
# removed from (added to) the 1p3/2 states goes to (comes from) the "other" states, which emit no gamma, the exact
# treatment also rescales the no-gamma events: w_p = 1 + x, w_none = 1 - x P_p,gamma / P_none (all selected events).
def f_pshell_exact(x):
    ws, wb = np.ones(len(cls_sig)), np.ones(len(cls_bkg))
    for w, cls, nuc in ((ws, cls_sig, nuc_sig), (wb, cls_bkg, nuc_bkg)):
        for n, t in TABLE_II.items():
            w[(nuc == n) & (cls == 1)] = 1 + x
            w[(nuc == n) & (cls == 0)] = 1 - x * t['p_gamma'] / t['none']
    return f_others(ws, wb) / F_CV - 1

P_EXACT = (100 * f_pshell_exact(-sigma_p_down), 100 * f_pshell_exact(+sigma_p_up))
print(f"p shell, exact population: fewer gammas ({-100*sigma_p_down:.2f}% strength) {P_EXACT[0]:+.2f}%, "
      f"more gammas (+{100*sigma_p_up:.2f}%) {P_EXACT[1]:+.2f}%   [CELL 6: +{SHELL['p'][0]:.2f} / -{SHELL['p'][1]:.2f}]")

p shell, exact population: fewer gammas (-16.42% strength) +9.23%, more gammas (+5.46%) -2.73%   [CELL 6: +10.90 / -3.16]


In [5]:
# Primary gamma-ray systematic for the budget (07): p shell and s shell from CELL 6, "others" one-sided (downward)
PRIMARY_UP   = np.sqrt(SHELL['p'][0]**2 + SHELL['s'][0]**2)
PRIMARY_DOWN = np.sqrt(SHELL['p'][1]**2 + SHELL['s'][1]**2 + (100 * min(OTHERS, 0.0))**2)

print("=" * 70)
print("  PRIMARY GAMMA-RAY SYSTEMATIC (% on the cross section)")
print("=" * 70)
print(f"  p shell            +{SHELL['p'][0]:5.2f} / -{SHELL['p'][1]:5.2f}")
print(f"  s shell            +{SHELL['s'][0]:5.2f} / -{SHELL['s'][1]:5.2f}")
print(f"  'other' states            / {100*OTHERS:+6.2f}   (one-sided)")
print(f"  total              +{PRIMARY_UP:5.2f} / -{PRIMARY_DOWN:5.2f}")

out = dict(
    provenance=X.provenance(dict(notebook='06_Primary_Dexcitation_Systematic.ipynb', mc=paths.MC_EMPIRICAL_DIR,
                                 inputs='results/08_systematics.json')),
    selected=dict(FV=N_sig_TOTAL_reco, OOFV_nu=N_bkg_TOTAL_reco, truth_FV=N_total_truth_FV,
                  categories_FV=cats_sig, categories_OOFV=cats_bkg),
    nuisances={n: dict(more_gammas=100 * per_nuisance['up'][n], fewer_gammas=100 * per_nuisance['down'][n])
               for n in NUISANCE_PARAMS},
    p_shell=list(SHELL['p']), s_shell=list(SHELL['s']),
    others=100 * OTHERS, others_mc_stat=100 * OTHERS_MCSTAT,
    primary_total=[float(PRIMARY_UP), float(PRIMARY_DOWN)],
    cross_checks=dict(v2_configuration=dict(p_shell=list(SHELL_OLD['p']), s_shell=list(SHELL_OLD['s']), others=100 * OTHERS_OLD),
                      p_shell_exact_population=list(P_EXACT), populations=POPULATIONS))
os.makedirs(paths.RESULTS_DIR, exist_ok=True)
with open(paths.RESULTS_DIR + '06_primary_gamma.json', 'w') as fh:
    json.dump(out, fh, indent=1, default=float)
print('\nwritten', paths.RESULTS_DIR + '06_primary_gamma.json')

  PRIMARY GAMMA-RAY SYSTEMATIC (% on the cross section)
  p shell            +10.90 / - 3.16
  s shell            + 1.61 / - 1.63
  'other' states            / -15.23   (one-sided)
  total              +11.01 / -15.64

written /workspace/hep/ANNIE/NCQE_XS_analysis_v3/results/06_primary_gamma.json
